In [1]:
import pickle
import sys
import os
sys.path.append(os.path.abspath('..'))
import pandas as pd
from config.settings import DATA_PATH_CLEAN, DATA_PATH_PROCESSED, ENCODER_PATH, SCALER_PATH
from config.functions import classify_label
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [2]:
data_clean = pd.read_parquet(DATA_PATH_CLEAN)
data_clean.loc[:, 'label'] = data_clean['loan_status'].apply(classify_label)
data_processed = data_clean.drop(columns = ['loan_status'])

In [3]:
drop_obj = ['issue_d', 'url', 'zip_code', 'addr_state', 'earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d']
num_cols = (data_processed.select_dtypes(include=['number']).columns).drop('label')
obj_cols = (data_processed.select_dtypes(include=['object']).columns).drop(drop_obj)

In [4]:
data_processed[obj_cols]

,term,grade,sub_grade,emp_length,home_ownership,verification_status,pymnt_plan,purpose,initial_list_status,application_type
0,36 months,B,B2,10+ years,RENT,Verified,n,credit_card,f,INDIVIDUAL
1,60 months,C,C4,< 1 year,RENT,Source Verified,n,car,f,INDIVIDUAL
2,36 months,C,C5,10+ years,RENT,Not Verified,n,small_business,f,INDIVIDUAL
3,36 months,C,C1,10+ years,RENT,Source Verified,n,other,f,INDIVIDUAL
4,60 months,B,B5,1 year,RENT,Source Verified,n,other,f,INDIVIDUAL
...,...,...,...,...,...,...,...,...,...,...
887374,36 months,B,B5,8 years,RENT,Verified,n,debt_consolidation,f,INDIVIDUAL
887375,36 months,B,B5,10+ years,MORTGAGE,Verified,n,home_improvement,f,INDIVIDUAL
887376,60 months,D,D2,5 years,RENT,Verified,n,debt_consolidation,w,INDIVIDUAL
887377,60 months,E,E3,1 year,RENT,Source Verified,n,debt_consolidation,w,INDIVIDUAL


In [5]:
encoder = OneHotEncoder()
scaler = StandardScaler()

In [6]:
data_obj = data_processed[obj_cols].copy()
data_obj_array = encoder.fit_transform(data_obj).toarray()
data_obj = pd.DataFrame(data_obj_array, columns=encoder.get_feature_names_out(data_obj.columns))

In [7]:
data_processed[num_cols]

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,...,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_amnt,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,total_rev_hi_lim
0,5000.0,5000.0,4975.0,10.65,162.87,24000.0,27.65,0.0,1.0,188.0,...,0.00,0.00,0.00,171.62,0.0,188.0,0.0,225.70261,139458.189336,32068.620045
1,2500.0,2500.0,2500.0,15.27,59.83,30000.0,1.00,0.0,5.0,188.0,...,0.00,117.08,1.11,119.66,0.0,188.0,0.0,225.70261,139458.189336,32068.620045
2,2400.0,2400.0,2400.0,15.96,84.33,12252.0,8.72,0.0,2.0,188.0,...,0.00,0.00,0.00,649.91,0.0,188.0,0.0,225.70261,139458.189336,32068.620045
3,10000.0,10000.0,10000.0,13.49,339.31,49200.0,20.00,0.0,1.0,35.0,...,16.97,0.00,0.00,357.48,0.0,188.0,0.0,225.70261,139458.189336,32068.620045
4,3000.0,3000.0,3000.0,12.69,67.79,80000.0,17.94,0.0,0.0,38.0,...,0.00,0.00,0.00,67.79,0.0,188.0,0.0,225.70261,139458.189336,32068.620045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887374,10000.0,10000.0,10000.0,11.99,332.10,31000.0,28.69,0.0,0.0,188.0,...,0.00,0.00,0.00,332.10,0.0,188.0,0.0,0.00000,25274.000000,17100.000000
887375,24000.0,24000.0,24000.0,11.99,797.03,79000.0,3.90,0.0,1.0,26.0,...,0.00,0.00,0.00,797.03,0.0,29.0,0.0,0.00000,140285.000000,10200.000000
887376,13000.0,13000.0,13000.0,15.99,316.07,35000.0,30.90,0.0,0.0,188.0,...,0.00,0.00,0.00,316.07,0.0,188.0,0.0,0.00000,34178.000000,18000.000000
887377,12000.0,12000.0,12000.0,19.99,317.86,64400.0,27.19,1.0,2.0,22.0,...,0.00,0.00,0.00,317.86,1.0,22.0,0.0,0.00000,58418.000000,27000.000000


In [8]:
data_num = data_processed[num_cols].copy()
data_num_array = scaler.fit_transform(data_num)
data_num = pd.DataFrame(data_num_array, columns=scaler.get_feature_names_out(data_num.columns))

In [9]:
data_processed_model = pd.concat([data_num,data_obj], axis=1)
data_processed_model['label'] = data_processed['label']
data_processed_model

,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,...,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,initial_list_status_f,initial_list_status_w,application_type_INDIVIDUAL,application_type_JOINT,label
0,-1.156460,-1.155635,-1.152256,-0.592611,-1.121467,-0.788703,0.552218,-0.364685,0.305857,0.957619,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0
1,-1.452829,-1.452198,-1.445430,0.461735,-1.543440,-0.695964,-0.998047,-0.364685,4.312143,0.957619,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1
2,-1.464683,-1.464061,-1.457275,0.619202,-1.443107,-0.970285,-0.548965,-0.364685,1.307428,0.957619,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0
3,-0.563724,-0.562507,-0.557025,0.055515,-0.398905,-0.399202,0.107207,-0.364685,0.305857,-0.992663,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0
4,-1.393555,-1.392886,-1.386203,-0.127055,-1.510842,0.076856,-0.012625,-0.364685,-0.695715,-0.954422,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887374,-0.563724,-0.562507,-0.557025,-0.286805,-0.428431,-0.680508,0.612716,-0.364685,-0.695715,0.957619,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0
887375,1.095939,1.098249,1.101329,-0.286805,1.475565,0.061399,-0.829350,-0.364685,0.305857,-1.107385,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0
887376,-0.208082,-0.206631,-0.201664,0.626049,-0.494078,-0.618682,0.741274,-0.364685,-0.695715,0.957619,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0
887377,-0.326629,-0.325257,-0.320117,1.538902,-0.486747,-0.164264,0.525459,0.795100,1.307428,-1.158373,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0


In [10]:
with open(ENCODER_PATH, 'wb') as file:
    pickle.dump(encoder, file)

with open(SCALER_PATH, 'wb') as file:
    pickle.dump(scaler, file)

In [11]:
#data_processed_model.to_csv(DATA_PATH_PROCESSED,sep=',', decimal='.', index=False)
data_processed_model.to_parquet(DATA_PATH_PROCESSED)